For two cases of very faint active regions (07-sep-18, 14-jan-21), we see suprising behavior in the consistency analysis: despite predicting relatively small amounts of material above 10 MK in the full-temperature range DEM, when the temperature range is restricted the DEM solution fails to be consistent with the NuSTAR inputs. Here we want to examine these outlier cases and explore whether this behavior seems like a real physical insight.

In [1]:
import pickle
import importlib
import glob
import os
import numpy as np

path_to_dodem = '/Users/jmdunca2/do-dem/'
from sys import path as sys_path
sys_path.append(path_to_dodem+'/dodem/')

import visualize_dem_results as viz


targets_file='/Users/jmdunca2/do-dem/reference_files/all_targets_postghost_postshut.pickle'
with open(targets_file, 'rb') as f:
    all_targets = pickle.load(f)

In [2]:
def checkresid(data, ind=-1):
    """
    Takes in DEM-result dictionary.
    
    Is the DEM-predicted value in the NuSTAR highest energy range consistent with the input value
    (within uncertainty)? 
    
    If yes, return 1 (If not, return 0).
    """
    
    hinu = data['dn_in'][ind]
    hiresid = data['dn_reg'][ind]/hinu 
    edn_lo = data['edn'][0][ind]
    edn_hi = data['edn'][1][ind]
    if hiresid-edn_lo <= 1 and hiresid+edn_hi >= 1:
        return 1
    else:
        return 0

In [6]:
keys = ['07-sep-18', '14-jan-21', '22-nov-21_1']

for kk in keys:
    filelist = all_targets[kk]['res_file_dict(s)'][0]['quiet files all-inst']
    for ff in filelist:
        #Get the consistency files (DEM results with different temp bounds)
        files = glob.glob(os.path.dirname(ff)+'/*'+kk+'_MC_DEM_result.pickle')
        sumcon=0
        for f in files:
            data, timestring, time = viz.load_DEM(f)
            print(data['dn_in'])
            sumcon += (1 if checkresid(data) == 0 else 0)

        print(sumcon)
        # sumcon=0
        # for f in files:
        #     data, timestring, time = viz.load_DEM(f)
        #     sumcon += (1 if checkresid(data, ind=-2) == 0 else 0)

        # print(sumcon)
        # sumcon=0
        # for f in files:
        #     data, timestring, time = viz.load_DEM(f)
        #     sumcon += (1 if checkresid(data, ind=-3) == 0 else 0)

        # print(sumcon)        
        print(ff)
        print('')

[0.9134769134503613, 6.135482196801453, 134.5103649574667, 293.6400381058051, 115.46421409582244, 5.374771954907007, 44.42583883220852, 1.0191898070441756, 0.03200200006133194]
[0.9134769134503613, 6.135482196801453, 134.5103649574667, 293.6400381058051, 115.46421409582244, 5.374771954907007, 44.42583883220852, 1.0191898070441756, 0.03200200006133194]
[0.9134769134503613, 6.135482196801453, 134.5103649574667, 293.6400381058051, 115.46421409582244, 5.374771954907007, 44.42583883220852, 1.0191898070441756, 0.03200200006133194]
[0.9134769134503613, 6.135482196801453, 134.5103649574667, 293.6400381058051, 115.46421409582244, 5.374771954907007, 44.42583883220852, 1.0191898070441756, 0.03200200006133194]
[0.9134769134503613, 6.135482196801453, 134.5103649574667, 293.6400381058051, 115.46421409582244, 5.374771954907007, 44.42583883220852, 1.0191898070441756, 0.03200200006133194]
3
/Users/jmdunca2/do-dem/DEM_folders//initial_dem_7sep18//14-56-55_15-14-20/14-56-55_15-14-20_5.6_7.2_07-sep-18_MC_

In [9]:
# Change to your local copy's location...
sys_path.append('/Users/jmdunca2/demreg/python/')
from dn2dem_pos import dn2dem_pos
import dn2dem_pos

mc_in=False
mc_rounds=10
reg_tweak=1.0
max_iter=10
gloci=1
rgt_fact=1.5 
dem_norm0=None
nmu=40
emd_int=True
emd_ret=True
rscl=False
rscl_factor=[]

#file1 = '/Users/jmdunca2/do-dem/DEM_folders//initial_dem_7sep18//18-27-45_18-36-30/18-27-45_18-36-30_5.6_7.2_07-sep-18_MC_DEM_result_withparams.pickle'
file1 = '/Users/jmdunca2/do-dem/DEM_folders//initial_dem_7sep18//18-27-45_18-36-30/18-27-45_18-36-30_5.6_7.2_07-sep-18_MC_DEM_result.pickle'

data, timestring, time = viz.load_DEM(file1)

import copy
new_in = copy.deepcopy(data['dn_in'])
print(new_in[-2])
new_in[-2] = 5.
#np.array(data['dn_in'])
demres=dn2dem_pos.dn2dem_pos(np.array(new_in), np.array(data['edn_in']), data['trmatrix'], data['ts_'], data['mts'], 
                                 gloci=gloci, emd_int=emd_int, emd_ret=emd_ret, 
                                 reg_tweak=reg_tweak, max_iter=max_iter, rgt_fact=rgt_fact, 
                               dem_norm0=dem_norm0, nmu=nmu)

dem70o,edem70o,elogt70o,chisq70o,dn_reg70o=demres

print(chisq70o)

0.8580463441017117
1.5178513461071381


Seems like having very few livetime-corrected counts/s above 6 keV is okay for DEM performance if there is a decent number of counts from 3.5-6 keV. Shall we double check that there are > 10 livetime corrected counts between 3.5-6keV?